# Hypothesis - Compare Top 20 and Bottom 20 Songs

Compare the characteristics of the most and least popular tracks

- Identify the top 20 tracks by popularity
- Identify the bottom 20 tracks by popularity
- Compare their audio features
- Create a comparison visualisation


## Inputs

CSV file used:

spotifydataset_Visualisation.csv


## Outputs

Plots for testing validity of each hypothesis before considering visualisation


## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"



In [1]:
#import libraries
import os
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import kruskal
#added by me for visualisation fine tuning
from matplotlib.ticker import MultipleLocator
from scipy.stats import f_oneway
from scipy.stats import pearsonr
from scipy.stats import spearmanr

#added by me for plotly.express visualisation issue
import nbformat 

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [ ]:
#DataFrame variables for EDA
dfSpotify_DataSet_Work = None
dfSpotify_DataSet_Temp = None
dfSpotify_DataSet_Temp1 = None

#stores current directory
strCurrentDir = ""

#other vars
dictDataFrames = dict()
fig = None
axis = None
intCount = 0
srtTemp = ""
strArtist = ""
strAlbum = ""
strTrack = ""

## Set Current Directory To Base Project Directory

In [3]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/rogerwilliams/Projects/Python/CourseProjects/Project3-Spotify-DirectWrite/Spotify-Music-Trend-Analysis


# Section 2

- Read csv file
- Look at hypothesis validity
- Use plot(s) to validate findings


## Read csv File Into Variable For Processing

In [4]:
#read csv file into DataFrame
dictDataFrames = modETL.funcReadVisualisationFilesReturnDictionary()
dfSpotify_DataSet = dictDataFrames["spotifydataset_Visualisation.csv"]

#create copy of the original DataFrame to work with
dfSpotify_DataSet_Work = dfSpotify_DataSet.copy()


1 csv Files Read Into DataFrames

DataFrames Created:
spotifydataset_Visualisation.csv




# Get Top 20 Songs By Popularity

In [5]:
#find top 20 songs by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#show results
dfSpotify_DataSet_Temp 

,Unnamed: 0.1,Unnamed: 0,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
20001,20001,20001,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),Unholy (feat. Kim Petras),100,156943,False,0.714,0.472,...,-7.375,1,0.0864,0.01300,0.000005,0.2660,0.238,131.121,4,dance
51664,51664,51664,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52","Quevedo: Bzrp Music Sessions, Vol. 52",99,198937,False,0.621,0.782,...,-5.548,1,0.0440,0.01250,0.033000,0.2300,0.550,128.033,4,hip-hop
89410,89411,89411,Manuel Turizo,La Bachata,La Bachata,98,162637,False,0.835,0.679,...,-5.329,0,0.0364,0.58300,0.000002,0.2180,0.850,124.980,4,reggaeton
81209,81210,81210,David Guetta;Bebe Rexha,I'm Good (Blue),I'm Good (Blue),98,175238,True,0.561,0.965,...,-3.673,0,0.0343,0.00383,0.000007,0.3710,0.304,128.040,4,pop
68303,68304,68304,Bad Bunny,Un Verano Sin Ti,Tití Me Preguntó,97,243716,False,0.650,0.715,...,-5.198,0,0.2530,0.09930,0.000291,0.1260,0.187,106.672,4,latino
68304,68305,68305,Bad Bunny;Chencho Corleone,Un Verano Sin Ti,Me Porto Bonito,97,178567,True,0.911,0.712,...,-5.105,0,0.0817,0.09010,0.000027,0.0933,0.425,92.005,4,latino
68358,68359,68359,Bad Bunny,Un Verano Sin Ti,Efecto,96,213061,False,0.801,0.475,...,-8.797,0,0.0516,0.14100,0.000017,0.0639,0.234,98.047,4,latino
81173,81174,81174,OneRepublic,I Ain’t Worried (Music From The Motion Picture...,I Ain't Worried,96,148485,False,0.704,0.797,...,-5.927,1,0.0475,0.08260,0.000745,0.0546,0.825,139.994,4,pop
20000,20000,20000,Chris Brown,Indigo (Extended),Under The Influence,96,184613,True,0.733,0.690,...,-5.529,0,0.0427,0.06350,0.000001,0.1050,0.310,116.992,4,dance
68352,68353,68353,Bad Bunny;Bomba Estéreo,Un Verano Sin Ti,Ojitos Lindos,95,258298,False,0.647,0.686,...,-5.745,0,0.0413,0.08000,0.000001,0.5280,0.268,79.928,4,latino


# Observations

As the dataset is a list of songs played over a predetermined time there are many duplicates for artist, album and track name so had
to remove the duplicates to get an accurate all round top 20

We can see that artist "Bad Bunny" is the most popular artist of the top 20 in terms of volume of entries, and the songs are all from the same album: "Un Verans Sin Ti"

We can also see than the value in the danceability column did not have a noticeable effect on the results

The top 2 songs also have the second and third highest tempo values
Interestingly all the top 20 songs were in the same time signature and the same key


# Plot The Results

In [6]:
#find top 20 songs by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(subset=["artists","album_name","track_name"], keep='first').head(20)

#show as sunburst plot
fig = px.sunburst(dfSpotify_DataSet_Temp, path=['album_name', 'artists', 'track_name'], values='popularity', color='popularity', 
                  color_continuous_scale='Viridis', title='Top 20 Songs by Popularity Sunburst Plot')
fig.update_layout(title={'text': 'Top 20 Songs by Popularity Sunburst Plot', 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},
                  height=800, width=1000)
#show album name, track and popularity on hover
fig.update_traces(hovertemplate='<b>Album:</b> %{label}<br><b>Track:</b> %{id}<br><b>Popularity:</b> %{value}<br><extra></extra>')
fig.show()

# EDA

## See If Any Correlation Between Certain Columns To Determine **Why** Song Is In The Top 20 List

Check correlations for:

- Popularity against duration_ms
- Popularity against energy
- Popularity against danceability
- Popularity against loudness
- Popularity against tempo
- Popularity against genre
- Popularity against explicit lyrics or not


In [7]:
#see if correlation between popularity and song duration

#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#delete duplicate duration_ms values
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["duration_ms"], keep='first')

#plotly scatter plot
# Create a scatter plot using Plotly
fig = px.scatter(dfSpotify_DataSet_Temp, x="popularity", y="duration_ms", title="Duration vs Popularity For Top 20 Songs")
fig.update_layout(title={"text": "Song Duration vs Popularity For Top 20 Songs", "x": 0.5, "xanchor": "center", "yanchor": "top"},
                  xaxis_title="Song Duration (ms) and Popularity",
                  yaxis_title="Frequency",
                  height=600, width=1000)
fig.show()



# Observation

We can see the X axis shows the scale of popularity (92 -100) from the dataset, and there is no correlation between song length and popularity

## Check For Correlation Between Popularity Against Energy

In [8]:
#check for correlation between popularity and energy

#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)


#delete duplicate energy values
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["energy"], keep='first')

#plotly scatter plot
# Create a scatter plot using Plotly
fig = px.scatter(dfSpotify_DataSet_Temp, x="popularity", y="energy", title="Energy vs Popularity For Top 20 Songs")
fig.update_layout(title={"text": "Song Energy vs Popularity For Top 20 Songs", "x": 0.5, "xanchor": "center", "yanchor": "top"},
                  xaxis_title="Song Energy and Popularity",
                  yaxis_title="Frequency",
                  height=600, width=1000)
fig.show()


# Observation

There a clear correlation between a songs energy rating and popularity with the majority of less popular songs falling below the 0.7 energy level

## Check For Correlation Between Popularity Against Danceability

In [9]:
#check for correlation between popularity against danceability

#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)


#delete duplicate danceability values
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["danceability"], keep='first')

#plotly scatter plot
# Create a scatter plot using Plotly
fig = px.scatter(dfSpotify_DataSet_Temp, x="popularity", y="danceability", title="Danceability vs Popularity For Top 20 Songs")
fig.update_layout(title={"text": "Song Danceability vs Popularity For Top 20 Songs", "x": 0.5, "xanchor": "center", "yanchor": "top"},
                  xaxis_title="Song Danceability and Popularity",
                  yaxis_title="Frequency",
                  height=600, width=1000)
fig.show()

# Observations

There a clear correlation between a songs danceability rating and popularity with the majority of less popular songs falling below the 0.7 danceability level\
which matches the energy comparision, so already we can see a correlation popularity combined with energy and danceability

## Check For Correlation Between Popularity Against Loudness

In [10]:
#check for correlation between popularity against loudness

#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)



#delete duplicate loudness values
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["loudness"], keep='first')

#plotly scatter plot
# Create a scatter plot using Plotly
fig = px.scatter(dfSpotify_DataSet_Temp, x="popularity", y="loudness", title="Loudness vs Popularity For Top 20 Songs")
fig.update_layout(title={"text": "Song Loudness vs Popularity For Top 20 Songs", "x": 0.5, "xanchor": "center", "yanchor": "top"},
                  xaxis_title="Song Loudness and Popularity",
                  yaxis_title="Frequency",
                  height=600, width=1000)
fig.show()

# Observations

We can see that quieter songs are actually more popular than louder ones

## Check For Correlation Between Popularity Against Tempo

In [11]:
#check for correlation between popularity against tempo

#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#delete duplicate tempo values
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["tempo"], keep='first')

#plotly scatter plot
# Create a scatter plot using Plotly
fig = px.scatter(dfSpotify_DataSet_Temp, x="popularity", y="tempo", title="Tempo vs Popularity For Top 20 Songs")
fig.update_layout(title={"text": "Song Tempo vs Popularity For Top 20 Songs", "x": 0.5, "xanchor": "center", "yanchor": "top"},
                  xaxis_title="Song Tempo and Popularity",
                  yaxis_title="Frequency",
                  height=600, width=1000)
fig.show()

# Observations

We can see that quicker songs are actually more popular than slower ones a pattern is emerging!

## Check For Correlation Between Popularity Against Genre

In [12]:
#check for correlation between popularity against genre
#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#delete duplicate genre values
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["track_genre"], keep='first')

#plotly scatter plot
# Create a scatter plot using Plotly
fig = px.scatter(dfSpotify_DataSet_Temp, x="popularity", y="track_genre", title="Track Genre vs Popularity For Top 20 Songs")
fig.update_layout(title={"text": "Song Genre vs Popularity For Top 20 Songs", "x": 0.5, "xanchor": "center", "yanchor": "top"},
                  xaxis_title="Song Genre and Popularity",
                  yaxis_title="Frequency",
                  height=600, width=1000)
fig.show()

# Observations

We can see that songs in the hip-hop and dance genres are the most popular

## Check For Correlation Between Popularity Against Explicit Lyrics Or Not

In [13]:
#check for correlation between popularity against explicit lyrics or not
#delete pure duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=False).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)


dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by='popularity', ascending=True)
#delete duplicate explicit values
#dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["explicit"], keep='first')

#plotly scatter plot
# Create a scatter plot using Plotly
fig = px.bar(dfSpotify_DataSet_Temp, x="popularity", y="explicit", title="Explicit Lyrics vs Popularity For Top 20 Songs")
fig.update_layout(title={"text": "Explicit Lyrics vs Popularity For Top 20 Songs", "x": 0.5, "xanchor": "center", "yanchor": "top"},
                  xaxis_title="Self-Explicit Lyrics and Popularity",
                  yaxis_title="Frequency",
                  height=600, width=1000)
fig.show()

# Observsations

We can see that songs without explicit lyrics are more popular

# Top 20 Songs By Popularity Conclusion

From the analysis we can confidently declare a "hit" needs to comprise of these attributes:

- Song duratin does not matter 
- High energy
- High danceability
- Quiet
- Fast tempo
- Be of Hip-Hop or Dance genre
- No explicit lyrics

# Get Top Bottom Songs By Popularity

In [14]:
#find bottom 20 songs by popularity
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)
#show results
dfSpotify_DataSet_Temp 

,Unnamed: 0.1,Unnamed: 0,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
56999,56999,56999,Håkan Hellström,LUGNA LÅTAR,Det kommer aldrig va över för mig,0,267106,False,0.554,0.877,...,-6.515,1,0.0469,0.01200,0.000039,0.4990,0.499,127.034,4,indie-pop
68345,68346,68346,Brray,Homecoming Latin Party,Bichotes Con Clase,0,193400,True,0.786,0.825,...,-5.191,0,0.2590,0.08720,0.003270,0.1470,0.752,180.026,4,latino
68346,68347,68347,Don Omar;Juan Magán,Perreo Tenebroso Vol. 4,No Sigue Modas Aka Ella No Sigue Modas,0,232000,False,0.685,0.862,...,-4.611,1,0.0627,0.00757,0.001400,0.0226,0.884,128.032,4,latino
19647,19647,19647,Tracy Lawrence,Chillin' It - Mellow Day Country,Just You and Me,0,220133,False,0.585,0.340,...,-8.433,0,0.0243,0.76700,0.000004,0.2540,0.260,100.379,4,country
68360,68361,68361,Chris Jedi;Ozuna;Brytiago,Perreo Tenebroso Vol. 4,Bipolar,0,220080,False,0.782,0.697,...,-5.850,1,0.0618,0.32000,0.000000,0.1440,0.416,76.031,4,latino
19645,19645,19645,Big & Rich;Bon Jovi,Chillin' It - Mellow Day Country,Born Again,0,234946,False,0.476,0.888,...,-4.926,1,0.0534,0.01310,0.000000,0.2770,0.617,162.000,4,country
19644,19644,19644,Steve Earle,Good Times Country,Go Amanda,0,214720,False,0.326,0.716,...,-3.547,1,0.0306,0.00227,0.001710,0.1230,0.648,143.148,4,country
19643,19643,19643,Tracy Lawrence,Country Car Hits,Excitable Boy,0,176760,False,0.628,0.854,...,-7.045,1,0.0343,0.08270,0.023200,0.1530,0.922,151.551,4,country
19642,19642,19642,Steve Earle,Finest Country,Jerusalem,0,236186,False,0.445,0.776,...,-4.492,1,0.0346,0.00252,0.007400,0.0981,0.403,118.379,4,country
19641,19641,19641,Sugarland,Christmas Country Songs 2022,Winter Wonderland,0,146973,False,0.587,0.876,...,-5.439,1,0.0432,0.12000,0.000000,0.0921,0.724,133.010,4,country


# Observations

As the dataset is a list of songs played over a predetermined time there are many duplicates for artist, album and track name so had
to remove the duplicates to get an accurate all round bottom 20

We can see an interesting statistic straight away, all the songs have a popularity rating of ZERO, because  there are 20 before we continue with the plot hypothesis as per the hypothesis requirement, for science let us see just how *many* zero popularity songs there are 

In [15]:
#find ALL songs with ZERO popularity 
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#show row count
print(f"There Are {dfSpotify_DataSet_Temp.shape[0]:,.2f} Unique Songs With Zero Popularity!")

dfSpotify_DataSet_Temp

There Are 20.00 Unique Songs With Zero Popularity!


,Unnamed: 0.1,Unnamed: 0,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
56999,56999,56999,Håkan Hellström,LUGNA LÅTAR,Det kommer aldrig va över för mig,0,267106,False,0.554,0.877,...,-6.515,1,0.0469,0.01200,0.000039,0.4990,0.499,127.034,4,indie-pop
68345,68346,68346,Brray,Homecoming Latin Party,Bichotes Con Clase,0,193400,True,0.786,0.825,...,-5.191,0,0.2590,0.08720,0.003270,0.1470,0.752,180.026,4,latino
68346,68347,68347,Don Omar;Juan Magán,Perreo Tenebroso Vol. 4,No Sigue Modas Aka Ella No Sigue Modas,0,232000,False,0.685,0.862,...,-4.611,1,0.0627,0.00757,0.001400,0.0226,0.884,128.032,4,latino
19647,19647,19647,Tracy Lawrence,Chillin' It - Mellow Day Country,Just You and Me,0,220133,False,0.585,0.340,...,-8.433,0,0.0243,0.76700,0.000004,0.2540,0.260,100.379,4,country
68360,68361,68361,Chris Jedi;Ozuna;Brytiago,Perreo Tenebroso Vol. 4,Bipolar,0,220080,False,0.782,0.697,...,-5.850,1,0.0618,0.32000,0.000000,0.1440,0.416,76.031,4,latino
19645,19645,19645,Big & Rich;Bon Jovi,Chillin' It - Mellow Day Country,Born Again,0,234946,False,0.476,0.888,...,-4.926,1,0.0534,0.01310,0.000000,0.2770,0.617,162.000,4,country
19644,19644,19644,Steve Earle,Good Times Country,Go Amanda,0,214720,False,0.326,0.716,...,-3.547,1,0.0306,0.00227,0.001710,0.1230,0.648,143.148,4,country
19643,19643,19643,Tracy Lawrence,Country Car Hits,Excitable Boy,0,176760,False,0.628,0.854,...,-7.045,1,0.0343,0.08270,0.023200,0.1530,0.922,151.551,4,country
19642,19642,19642,Steve Earle,Finest Country,Jerusalem,0,236186,False,0.445,0.776,...,-4.492,1,0.0346,0.00252,0.007400,0.0981,0.403,118.379,4,country
19641,19641,19641,Sugarland,Christmas Country Songs 2022,Winter Wonderland,0,146973,False,0.587,0.876,...,-5.439,1,0.0432,0.12000,0.000000,0.0921,0.724,133.010,4,country


# Create A Meaningful Plot 

With 9,382 unique songs with ZERO popularity that clearly is pointless to plot (is there a plot big enough?)

See if same with values of one

In [16]:
#find ALLsongs with popularity of 1
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work[ dfSpotify_DataSet_Work["popularity"] == 1]

#delete duplicates
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.drop_duplicates(subset=["artists","album_name","track_name"], keep='first')

#show row count
print(f"There Are {dfSpotify_DataSet_Temp.shape[0]:,.2f} Unique Songs With Zero Popularity!")

dfSpotify_DataSet_Temp

There Are 1,153.00 Unique Songs With Zero Popularity!


,Unnamed: 0.1,Unnamed: 0,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
25,25,25,Jason Mraz,Mellow Adult Pop,Bella Luna,1,302346,False,0.755,0.454,...,-9.609,0,0.0352,0.75700,0.000000,0.2360,0.330,120.060,4,acoustic
2019,2019,2019,Red Hot Chili Peppers,Tek It - New Noise,Black Summer,1,232412,False,0.417,0.682,...,-5.071,1,0.0295,0.02670,0.000669,0.1090,0.351,105.542,4,alt-rock
2028,2028,2028,The Killers;Armin van Buuren;Benno De Goeij,Beats Electro Mood,Human - Armin van Buuren Dub Remix,1,445000,False,0.640,0.821,...,-5.789,0,0.0732,0.00122,0.145000,0.2030,0.624,135.478,4,alt-rock
2029,2029,2029,The Killers,Kick It,Mr. Brightside,1,223973,False,0.344,0.931,...,-3.759,1,0.0783,0.00104,0.000000,0.0903,0.251,148.104,4,alt-rock
2031,2031,2031,Red Hot Chili Peppers,Kick It,Fire - Remastered,1,123800,True,0.421,0.986,...,-1.795,0,0.1520,0.00482,0.000000,0.2950,0.263,105.830,4,alt-rock
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113035,113036,113036,Chris Tomlin;Lauren Daigle,Christmas Dinner,Noel - Live,1,257720,False,0.349,0.261,...,-9.478,0,0.0306,0.74800,0.000000,0.1040,0.103,137.954,4,world-music
113371,113372,113372,Michael W. Smith,Christmas at Home,Christmas Is Here,1,206165,False,0.219,0.281,...,-9.790,1,0.0322,0.79600,0.000000,0.0773,0.301,183.446,3,world-music
113372,113373,113373,Michael W. Smith,Christmas at Home,Christmas Is Here - Radio Edit,1,181143,False,0.226,0.279,...,-8.730,1,0.0299,0.73800,0.000000,0.1410,0.354,183.600,3,world-music
113374,113375,113375,Michael W. Smith,Christmas at Home,God with God,1,250859,False,0.226,0.388,...,-8.440,1,0.0314,0.53500,0.000000,0.1030,0.158,114.126,4,world-music


# Observations

Same issue when looking for popularity of 1, 116 unique records so will stick to the hypothesis and look at popularity of zero as that is the lowest score

In [17]:
#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#sort dataset by popularity in ascending order
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.sort_values(by='popularity', ascending=True)

#show results
dfSpotify_DataSet_Temp.head(20)

,Unnamed: 0.1,Unnamed: 0,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
56999,56999,56999,Håkan Hellström,LUGNA LÅTAR,Det kommer aldrig va över för mig,0,267106,False,0.554,0.877,...,-6.515,1,0.0469,0.01200,0.000039,0.4990,0.499,127.034,4,indie-pop
19632,19632,19632,Don Henley,Обратно в клас - rock,The Boys Of Summer,0,288333,False,0.516,0.775,...,-6.409,1,0.0412,0.39600,0.001290,0.2100,0.858,177.401,4,country
19633,19633,19633,Devin Dawson,Laidback Country,All on Me,0,224626,False,0.634,0.666,...,-5.489,1,0.0290,0.13900,0.000000,0.2160,0.625,80.989,4,country
19634,19634,19634,Big & Rich;Bon Jovi,Country Road Songs,Born Again,0,234946,False,0.476,0.888,...,-4.926,1,0.0534,0.01310,0.000000,0.2770,0.617,162.000,4,country
68362,68363,68363,Cali Y El Dandee;Sebastian Yatra,Hora del taco sin auto,Locura,0,209440,False,0.758,0.778,...,-3.562,1,0.0657,0.08060,0.000000,0.1980,0.532,92.983,4,latino
19637,19637,19637,Easton Corbin,Mientras hago aromaterapia,A Girl Like You,0,217840,False,0.733,0.768,...,-4.481,1,0.0354,0.01970,0.000001,0.0643,0.830,105.006,4,country
19638,19638,19638,Big & Rich,Country Christmas Time,Blue Christmas,0,177165,False,0.514,0.901,...,-5.674,1,0.1140,0.09150,0.000015,0.0760,0.422,120.647,4,country
19639,19639,19639,Big & Rich,Country Holiday,Blue Christmas,0,177165,False,0.514,0.901,...,-5.674,1,0.1140,0.09150,0.000015,0.0760,0.422,120.647,4,country
19640,19640,19640,Glen Campbell,(60's) Sixties Collected Volume 2,Gentle On My Mind - Remastered 2001,0,178626,False,0.597,0.415,...,-13.799,1,0.0323,0.18900,0.000000,0.1220,0.826,108.987,4,country
19641,19641,19641,Sugarland,Christmas Country Songs 2022,Winter Wonderland,0,146973,False,0.587,0.876,...,-5.439,1,0.0432,0.12000,0.000000,0.0921,0.724,133.010,4,country


# Observations

Due to the results containing popularity values of *zero* only no point putting in a conventional plot so will use sunburst again!

# Plot The Results

In [18]:
#show as sunburst plot

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

dfSpotify_DataSet_Temp["count"] = 1

fig = px.sunburst(dfSpotify_DataSet_Temp.head(20), path=["artists", "album_name", "track_name"], values="count", color="popularity", 
                  color_continuous_scale='Viridis', title='Bottom 20 Songs by Popularity Sunburst Plot')
fig.update_layout(title={'text': 'Bottom 20 Songs by Popularity Sunburst Plot', 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},
                  height=800, width=1000)
#show album name, track and popularity on hover
fig.update_traces(hovertemplate='<b>Album:</b> %{label}<br><b>Track:</b> %{id}<br><extra></extra>')
fig.show()

# Observations

Nat King Cole is not very popular on Spotify! Maybe this popularity rating is due to a lot of his most popular songs are seasonal

# EDA

## See If Any Correlation Between Certain Columns To Determine **Why** Song Is In The Bottom 20 List

Because all the bottom 20 songs have a popularity of *one* no point there is noting to correlate that would not bias the outcome

So instead will look at: 

- Artist
- Genre
- Album
- Explicit lyrics or not
- loudness
- energy
- tempo
- danceability

In [19]:
#get count of track_name columns to attempt to determine *why* thes songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for artist, album, and track name
dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp[["artists", "album_name", "track_name"]].drop_duplicates()

#now sort by album name, then artist, then track name
dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp1.sort_values(by=["artists", "album_name", "track_name"], ascending=[True, True, True])

#get how many songs per artist have popularity of ZERO
dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp1.groupby("artists").agg({"track_name": "count"}).reset_index()

#sort by track_name
dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp1.sort_values(by=["track_name"], ascending=[False])

print("DataFrame Showing Most Unpoplar Artists With Songs With Popularity of ZERO")
print("=" * len("DataFrame Showing Most Unpoplar Artists With Songs With Popularity of ZERO"))
print(dfSpotify_DataSet_Temp1)
print("Note: track_name column shows the number of songs per artist with popularity of ZERO")

DataFrame Showing Most Unpoplar Artists With Songs With Popularity of ZERO
                             artists  track_name
0                         Big & Rich           2
1                Big & Rich;Bon Jovi           2
5                       Devin Dawson           2
12                       Steve Earle           2
14                    Tracy Lawrence           2
2                              Brray           1
3   Cali Y El Dandee;Sebastian Yatra           1
4          Chris Jedi;Ozuna;Brytiago           1
6                         Don Henley           1
7                           Don Omar           1
8                Don Omar;Juan Magán           1
9                      Easton Corbin           1
10                     Glen Campbell           1
11                   Håkan Hellström           1
13                         Sugarland           1
Note: track_name column shows the number of songs per artist with popularity of ZERO


# Observations

Big & Rich are not very popular, lets repeat the process for genre

In [20]:
#get genre column to attempt to determine *why* these songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for artist, album, and track name
dfSpotify_DataSet_Temp1 = dfSpotify_DataSet_Temp[["artists", "track_genre"]].drop_duplicates()

#now sort by genre, then artist
dfSpotify_DataSet_Temp1  = dfSpotify_DataSet_Temp1.sort_values(by=["track_genre", "artists"], ascending=[True, True])

print("DataFrame Showing Most Unpoplar Genre Artists With Songs With Popularity of ZERO")
print("=" * len("DataFrame Showing Most Unpoplar Genre Artists With Songs With Popularity of ZERO"))
print(dfSpotify_DataSet_Temp1)


DataFrame Showing Most Unpoplar Genre Artists With Songs With Popularity of ZERO
                                artists track_genre
19639                        Big & Rich     country
19645               Big & Rich;Bon Jovi     country
19633                      Devin Dawson     country
19632                        Don Henley     country
19637                     Easton Corbin     country
19640                     Glen Campbell     country
19644                       Steve Earle     country
19641                         Sugarland     country
19647                    Tracy Lawrence     country
56999                   Håkan Hellström   indie-pop
68345                             Brray      latino
68362  Cali Y El Dandee;Sebastian Yatra      latino
68360         Chris Jedi;Ozuna;Brytiago      latino
68344                          Don Omar      latino
68346               Don Omar;Juan Magán      latino


# Observations

Clearly country music is the top of the least popular song styles in this data sample

In [21]:
#get album column to attempt to determine *why* these songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for artist, album, and track name
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[[ "album_name", "artists"]].drop_duplicates()

#group by album name and artist
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby(["album_name", "artists"]).size().reset_index(name='count')

#now sort by album name
dfSpotify_DataSet_Temp  = dfSpotify_DataSet_Temp.sort_values(by=["album_name"], ascending=[False])

print("DataFrame Showing Most Unpoplar Albums With Songs With Popularity of ZERO")
print("=" * len("DataFrame Showing Most Unpoplar Albums With Songs With Popularity of ZERO"))
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["album_name", "artists"]]
print(dfSpotify_DataSet_Temp)


DataFrame Showing Most Unpoplar Albums With Songs With Popularity of ZERO
                           album_name                           artists
19              Обратно в клас - rock                        Don Henley
18            Perreo Tenebroso Vol. 4               Don Omar;Juan Magán
17            Perreo Tenebroso Vol. 4         Chris Jedi;Ozuna;Brytiago
16         Mientras hago aromaterapia                     Easton Corbin
15                   Laidback Country                      Devin Dawson
14                        LUGNA LÅTAR                   Håkan Hellström
13             Hora del taco sin auto  Cali Y El Dandee;Sebastian Yatra
11             Homecoming Latin Party                             Brray
12             Homecoming Latin Party                          Don Omar
10                 Good Times Country                       Steve Earle
9                  Good Times Country                      Devin Dawson
8                      Finest Country                       St

# Observations

No clear winner in the most unpoplar album stakes

In [22]:
#get explicit column to attempt to determine *why* these songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for artist, album, and track name
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[[ "album_name", "artists", "explicit"]].drop_duplicates()

#group by explicit, album name and artist
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby(["explicit","album_name", "artists"]).size().reset_index(name='count')

#now sort by explicit, album name, then artist
dfSpotify_DataSet_Temp  = dfSpotify_DataSet_Temp.sort_values(by=["explicit", "album_name", "artists"], ascending=[False, False, False])

print("DataFrame Showing Most Unpoplar Songs With Popularity of ZERO (Explicit Lyrics or Not)")
print("=" * len("DataFrame Showing Most Unpoplar Songs With Popularity of ZERO (Explicit Lyrics or Not)"))
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["album_name", "artists", "explicit"]]
print(dfSpotify_DataSet_Temp)


DataFrame Showing Most Unpoplar Songs With Popularity of ZERO (Explicit Lyrics or Not)
                           album_name                           artists  \
19             Homecoming Latin Party                             Brray   
18              Обратно в клас - rock                        Don Henley   
17            Perreo Tenebroso Vol. 4               Don Omar;Juan Magán   
16            Perreo Tenebroso Vol. 4         Chris Jedi;Ozuna;Brytiago   
15         Mientras hago aromaterapia                     Easton Corbin   
14                   Laidback Country                      Devin Dawson   
13                        LUGNA LÅTAR                   Håkan Hellström   
12             Hora del taco sin auto  Cali Y El Dandee;Sebastian Yatra   
11             Homecoming Latin Party                          Don Omar   
10                 Good Times Country                       Steve Earle   
9                  Good Times Country                      Devin Dawson   
8            

# Observations

We can see that explicit lyrics are not a contributing factor to a songs lack of popularity

In [23]:
#get loudness column to attempt to determine *why* these songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for artist, album, and track name
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[[ "album_name", "artists", "loudness"]].drop_duplicates()

#group by loudness, album name and artist
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby(["loudness","album_name", "artists"]).size().reset_index(name='count')

#now sort by loudness, album name, then artist, then genre
dfSpotify_DataSet_Temp  = dfSpotify_DataSet_Temp.sort_values(by=["loudness", "album_name", "artists"], ascending=[False, False, False])

print("DataFrame Showing Most Unpoplar Songs By Loudness With Popularity of ZERO")
print("=" * len("DataFrame Showing Most Unpoplar Songs By Loudness With Popularity of ZERO"))
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["album_name", "artists", "loudness"]]
print(dfSpotify_DataSet_Temp)


DataFrame Showing Most Unpoplar Songs By Loudness With Popularity of ZERO
                           album_name                           artists  \
19                 Good Times Country                       Steve Earle   
18             Hora del taco sin auto  Cali Y El Dandee;Sebastian Yatra   
17         Mientras hago aromaterapia                     Easton Corbin   
16                     Finest Country                       Steve Earle   
15            Perreo Tenebroso Vol. 4               Don Omar;Juan Magán   
14                 Country Road Songs               Big & Rich;Bon Jovi   
13   Chillin' It - Mellow Day Country               Big & Rich;Bon Jovi   
12             Homecoming Latin Party                             Brray   
11       Christmas Country Songs 2022                         Sugarland   
10                   Laidback Country                      Devin Dawson   
9                     Country Holiday                        Big & Rich   
8              Country Chr

# Observations

Mean loudness is around 5.400 which is a *potential* insight

In [24]:
#get energy column to attempt to determine *why* these songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for artist, album, and track name
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[[ "album_name", "artists", "energy"]].drop_duplicates()

#group by energy, album name and artist
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby(["energy","album_name", "artists"]).size().reset_index(name='count')

#now sort by energy, album name, then artist
dfSpotify_DataSet_Temp  = dfSpotify_DataSet_Temp.sort_values(by=["energy", "album_name", "artists"], ascending=[False, False, False])

print("DataFrame Showing Most Unpoplar Songs By Energy With Popularity of ZERO")
print("=" * len("DataFrame Showing Most Unpoplar Songs By Energy With Popularity of ZERO"))
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["album_name", "artists", "energy"]]
print(dfSpotify_DataSet_Temp)


DataFrame Showing Most Unpoplar Songs By Energy With Popularity of ZERO
                           album_name                           artists  \
19                    Country Holiday                        Big & Rich   
18             Country Christmas Time                        Big & Rich   
17                 Country Road Songs               Big & Rich;Bon Jovi   
16   Chillin' It - Mellow Day Country               Big & Rich;Bon Jovi   
15                        LUGNA LÅTAR                   Håkan Hellström   
14       Christmas Country Songs 2022                         Sugarland   
13            Perreo Tenebroso Vol. 4               Don Omar;Juan Magán   
12                   Country Car Hits                    Tracy Lawrence   
11             Homecoming Latin Party                             Brray   
10             Hora del taco sin auto  Cali Y El Dandee;Sebastian Yatra   
9                      Finest Country                       Steve Earle   
8               Обратно в кл

# Observations

No real insights here either!

In [25]:
#get tempo column to attempt to determine *why* these songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for artist, album, and track name
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[[ "album_name", "artists", "tempo"]].drop_duplicates()

#group by tempo, album name and artist
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby(["tempo","album_name", "artists"]).size().reset_index(name='count')

#now sort by tempo, album name, then artist, then genre
dfSpotify_DataSet_Temp  = dfSpotify_DataSet_Temp.sort_values(by=["tempo", "album_name", "artists"], ascending=[False, False, False])

print("DataFrame Showing Most Unpoplar Songs By Tempo With Popularity of ZERO")
print("=" * len("DataFrame Showing Most Unpoplar Songs By Tempo With Popularity of ZERO"))
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["album_name", "artists", "tempo"]]
print(dfSpotify_DataSet_Temp)


DataFrame Showing Most Unpoplar Songs By Tempo With Popularity of ZERO
                           album_name                           artists  \
19             Homecoming Latin Party                             Brray   
18              Обратно в клас - rock                        Don Henley   
17                 Country Road Songs               Big & Rich;Bon Jovi   
16   Chillin' It - Mellow Day Country               Big & Rich;Bon Jovi   
15                   Country Car Hits                    Tracy Lawrence   
14                 Good Times Country                       Steve Earle   
13       Christmas Country Songs 2022                         Sugarland   
12            Perreo Tenebroso Vol. 4               Don Omar;Juan Magán   
11                        LUGNA LÅTAR                   Håkan Hellström   
10                    Country Holiday                        Big & Rich   
9              Country Christmas Time                        Big & Rich   
8                      Finest

# Observations

Now we see something, tempos are almost exlusively over 100BPM - finally something to work with!

In [26]:
#get danceability column to attempt to determine *why* these songs are so unpopular

#filter the dataset for records with popularity of ZERO
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Work.sort_values(by='popularity', ascending=True).drop_duplicates(
    subset=["artists","album_name","track_name"], keep='first').head(20)

#create dataframe for album, artist and danceability
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[[ "album_name", "artists", "danceability"]].drop_duplicates()

#group by danceability, album name and artist
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp.groupby(["danceability","album_name", "artists"]).size().reset_index(name='count')


#now sort by danceability, album name, then artist, then genre
dfSpotify_DataSet_Temp  = dfSpotify_DataSet_Temp.sort_values(by=["danceability", "album_name", "artists"], ascending=[False, False, False])

print("DataFrame Showing Most Unpoplar Songs By Danceability With Popularity of ZERO")
print("=" * len("DataFrame Showing Most Unpoplar Songs By Danceability With Popularity of ZERO"))
dfSpotify_DataSet_Temp = dfSpotify_DataSet_Temp[["album_name", "artists", "danceability"]]
print(dfSpotify_DataSet_Temp)


DataFrame Showing Most Unpoplar Songs By Danceability With Popularity of ZERO
                           album_name                           artists  \
19             Homecoming Latin Party                          Don Omar   
18             Homecoming Latin Party                             Brray   
17            Perreo Tenebroso Vol. 4         Chris Jedi;Ozuna;Brytiago   
16             Hora del taco sin auto  Cali Y El Dandee;Sebastian Yatra   
15         Mientras hago aromaterapia                     Easton Corbin   
14            Perreo Tenebroso Vol. 4               Don Omar;Juan Magán   
13                 Good Times Country                      Devin Dawson   
12                   Laidback Country                      Devin Dawson   
11                   Country Car Hits                    Tracy Lawrence   
10  (60's) Sixties Collected Volume 2                     Glen Campbell   
9        Christmas Country Songs 2022                         Sugarland   
8    Chillin' It - Mel

# Observations

Danceability is has a mean around 0.550 which gives a little more insight as too why these songs are so unpopular

# Conclusion

We can assume from the sample data that what makes a song unpopular is:

- Genre of Country
- Tempo about 100BPM
- Loudness of 5/6 out of 10

Bad day for bopping country music!